# 🚀 Qwen-Image-2.1 Uncensored (Q4_0 GGUF) 실시간 고속 검증 생성기

요청하신 **[abenzerps/Qwen-Image-2.1-Uncensored-GGUF](https://huggingface.co/abenzerps/Qwen-Image-2.1-Uncensored-GGUF)** 모델 전용 고속 검증 노트북입니다.

---
### ⚡ 주요 최적화 사항
1. **16GB 가상 메모리(SWAP) 자동 활성화**: 9.3GB 대형 텍스트 인코더 로딩 시 Colab 12.7GB 시스템 램 한도 초과 및 프로세스 튕김 원천 차단.
2. **ComfyUI Dynamic VRAM 가속**: 속도를 심각하게 떨어뜨리던 `--lowvram`을 제거하고 4.15GB GGUF 디퓨전 모델을 T4 16GB VRAM에 직접 상주시킵니다.
3. **실시간 단계별 로그 모니터링**: 백엔드에서 실제로 몇 번째 스텝을 계산 중인지 실시간으로 웹UI 진행창에 표시됩니다.

👉 **상단 메뉴에서 `런타임 > 모두 실행(Run all)`**을 클릭하시면 모든 과정이 전자동으로 진행됩니다.

### 1단계: GPU 환경 확인
Colab 상단 메뉴 `런타임 > 런타임 유형 변경`에서 **T4 GPU**로 설정되어 있는지 확인합니다.

In [ ]:
!nvidia-smi

### 2단계: 시스템 가상 메모리 확보 및 ComfyUI-GGUF 엔진 설치

In [ ]:
# 1. 기존 잔여 GPU 프로세스 정리 (메모리 100% 확보)
!fuser -k -9 /dev/nvidia* > /dev/null 2>&1 || true
!pkill -9 -f "main.py" > /dev/null 2>&1 || true

# 2. 16GB 가상 메모리(SWAP) 생성 (14GB 대형 모델 안전 로딩용)
print('💾 16GB 가상 메모리(SWAP) 설정 중...')
!fallocate -l 16G /swapfile 2>/dev/null && chmod 600 /swapfile && mkswap /swapfile 2>/dev/null && swapon /swapfile 2>/dev/null || true
!free -h

# 3. 필수 라이브러리 및 Gradio 설치
!apt-get update -qq && apt-get install -y -qq aria2
!pip install -q gradio Pillow gguf huggingface_hub

# 4. ComfyUI 백엔드 클론 및 설치
%cd /content
!git clone https://github.com/comfyanonymous/ComfyUI.git 2>/dev/null || true
%cd /content/ComfyUI
!pip install -q -r requirements.txt

# 5. Qwen-Image-2.1 네이티브 지원 GGUF 노드 설치
%cd /content/ComfyUI/custom_nodes
!git clone https://github.com/leejet/ComfyUI-GGUF.git 2>/dev/null || true
%cd /content/ComfyUI/custom_nodes/ComfyUI-GGUF
!pip install -q -r requirements.txt
%cd /content/ComfyUI
print('✅ 환경 설치 완료!')

### 3단계: Qwen-Image-2.1 Uncensored 모델 파일 다운로드
이미 다운로드되어 있다면 즉시 통과합니다.

In [ ]:
import os

os.makedirs('/content/ComfyUI/models/diffusion_models', exist_ok=True)
os.makedirs('/content/ComfyUI/models/unet', exist_ok=True)
os.makedirs('/content/ComfyUI/models/text_encoders', exist_ok=True)
os.makedirs('/content/ComfyUI/models/clip', exist_ok=True)
os.makedirs('/content/ComfyUI/models/vae', exist_ok=True)

ua = '--user-agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"'

# 1. Qwen-Image-2.1 UC Q4_0 GGUF (4.15 GB)
diff_path = '/content/ComfyUI/models/diffusion_models/qwen-image-2.1-UC-Q4_0.gguf'
if not os.path.exists(diff_path) or os.path.getsize(diff_path) < 1000000:
    print('📥 [1/3] Qwen-Image-2.1 UC Q4_0 GGUF 다운로드 중 (약 4.15 GB)...')
    !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M $ua \
      'https://huggingface.co/abenzerps/Qwen-Image-2.1-Uncensored-GGUF/resolve/main/qwen-image-2.1-UC-Q4_0.gguf' \
      -d /content/ComfyUI/models/diffusion_models -o qwen-image-2.1-UC-Q4_0.gguf
else:
    print('✅ [1/3] Qwen-Image-2.1 UC Q4_0 GGUF 준비 완료.')

!ln -sf /content/ComfyUI/models/diffusion_models/qwen-image-2.1-UC-Q4_0.gguf /content/ComfyUI/models/unet/ 2>/dev/null || true

# 2. Qwen3-VL 8B INT8 ConvRot Text Encoder (9.35 GB)
clip_path = '/content/ComfyUI/models/text_encoders/qwen3vl_8b_int8_convrot.safetensors'
if not os.path.exists(clip_path) or os.path.getsize(clip_path) < 1000000:
    print('📥 [2/3] Qwen3-VL 8B INT8 Text Encoder 다운로드 중 (약 9.35 GB)...')
    !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M $ua \
      'https://huggingface.co/abenzerps/Qwen-Image-2.1-Uncensored-GGUF/resolve/main/text_encoders/qwen3vl_8b_int8_convrot.safetensors' \
      -d /content/ComfyUI/models/text_encoders -o qwen3vl_8b_int8_convrot.safetensors
else:
    print('✅ [2/3] Qwen3-VL 8B INT8 Text Encoder 준비 완료.')

!ln -sf /content/ComfyUI/models/text_encoders/qwen3vl_8b_int8_convrot.safetensors /content/ComfyUI/models/clip/ 2>/dev/null || true

# 3. Qwen-Image-2.1 VAE (676 MB)
vae_path = '/content/ComfyUI/models/vae/qwen_image_2.1_vae_bf16.safetensors'
if not os.path.exists(vae_path) or os.path.getsize(vae_path) < 1000000:
    print('📥 [3/3] Qwen-Image-2.1 VAE 다운로드 중 (약 676 MB)...')
    !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M $ua \
      'https://huggingface.co/abenzerps/Qwen-Image-2.1-Uncensored-GGUF/resolve/main/vae/qwen_image_2.1_vae_bf16.safetensors' \
      -d /content/ComfyUI/models/vae -o qwen_image_2.1_vae_bf16.safetensors
else:
    print('✅ [3/3] Qwen-Image-2.1 VAE 준비 완료.')

print('🎉 모든 Qwen 2.1 모델 준비 완료!')

### 4단계: Qwen-Image-2.1 고속 웹 생성기 실행
- 실행 즉시 화면 아래에 직관적인 웹 폼이 열립니다.
- 실시간 진행 로그가 웹UI 진행창에 표시되어 현재 몇 번째 스텝인지 확인할 수 있습니다.

In [ ]:
import subprocess
import time
import urllib.request
import urllib.parse
import urllib.error
import json
import random
import io
import re
import os
import gradio as gr
from PIL import Image

%cd /content/ComfyUI

# 1. 기존 프로세스 종료 및 포트 8188 청소
!fuser -k -9 8188/tcp > /dev/null 2>&1 || true
!pkill -9 -f "main.py" > /dev/null 2>&1 || true
time.sleep(2)

# 2. 고속 백그라운드 ComfyUI 엔진 실행 (Dynamic VRAM 활용으로 풀스피드 가속)
print('⏳ Qwen-Image-2.1 백엔드 엔진 시작 중...')
log_path = '/content/comfyui.log'
log_file = open(log_path, 'w')
engine_proc = subprocess.Popen(
    ['python', 'main.py', '--listen', '127.0.0.1', '--port', '8188', '--preview-method', 'none'],
    stdout=log_file,
    stderr=log_file
)

# 엔진 준비 대기 (최대 40초)
ready = False
for i in range(20):
    try:
        with urllib.request.urlopen('http://127.0.0.1:8188/system_stats', timeout=2) as resp:
            if resp.status == 200:
                print('✅ Qwen-Image 백엔드 엔진 준비 완료!')
                ready = True
                break
    except Exception:
        time.sleep(2)

if not ready:
    print('⚠️ 백엔드 시작 로그:')
    if os.path.exists(log_path):
        with open(log_path) as f:
            print(f.read()[-1000:])

# 로그 파일에서 최신 진행 상태 파싱 함수
def get_latest_log_status():
    if not os.path.exists(log_path):
        return '대기 중...'
    try:
        with open(log_path, 'r', encoding='utf-8', errors='ignore') as f:
            lines = f.readlines()
        if not lines:
            return '대기 중...'
        # 최근 15개 라인 역순 탐색
        for line in reversed(lines[-15:]):
            line_str = line.strip()
            if '%' in line_str and ('/' in line_str or 'it/s' in line_str or 's/it' in line_str):
                return f'🎨 디퓨전 연산 중: {line_str}'
            if 'Loading' in line_str or 'model weight' in line_str:
                return f'⏳ 모델 가중치 로드 중: {line_str}'
            if 'prompt' in line_str or 'Requested' in line_str:
                return f'⚙️ 작업 처리 중: {line_str}'
        return lines[-1].strip()[:80]
    except Exception:
        return '연산 처리 중...'

# 3. Qwen-Image-2.1 이미지 생성 함수
def generate_image(prompt, negative_prompt, steps, cfg, width, height, seed, progress=gr.Progress()):
    if not prompt or prompt.strip() == '':
        raise gr.Error('프롬프트를 입력해 주세요!')
    
    if seed == -1 or seed is None:
        seed = random.randint(1, 1000000000)
    
    # 백엔드 생존 확인
    if engine_proc.poll() is not None:
        tail_log = ''
        if os.path.exists(log_path):
            with open(log_path) as f:
                tail_log = ''.join(f.readlines()[-20:])
        raise gr.Error(f'백엔드 엔진이 종료되었습니다 (코드: {engine_proc.returncode}). 오류 로그:\n{tail_log}')
    
    progress(0.05, desc='⏳ [1단계] Qwen-Image-2.1 파이프라인 요청 전송 중...')
    
    workflow = {
        '1': {'class_type': 'UnetLoaderGGUF', 'inputs': {'unet_name': 'qwen-image-2.1-UC-Q4_0.gguf'}},
        '2': {'class_type': 'CLIPLoader', 'inputs': {'clip_name': 'qwen3vl_8b_int8_convrot.safetensors', 'type': 'qwen_image'}},
        '3': {'class_type': 'VAELoader', 'inputs': {'vae_name': 'qwen_image_2.1_vae_bf16.safetensors'}},
        '4': {'class_type': 'CLIPTextEncode', 'inputs': {'clip': ['2', 0], 'text': prompt}},
        '5': {'class_type': 'CLIPTextEncode', 'inputs': {'clip': ['2', 0], 'text': negative_prompt}},
        '6': {'class_type': 'EmptyLatentImage', 'inputs': {'width': int(width), 'height': int(height), 'batch_size': 1}},
        '7': {'class_type': 'KSampler', 'inputs': {
            'model': ['1', 0],
            'positive': ['4', 0],
            'negative': ['5', 0],
            'latent_image': ['6', 0],
            'seed': int(seed),
            'steps': int(steps),
            'cfg': float(cfg),
            'sampler_name': 'euler',
            'scheduler': 'normal',
            'denoise': 1.0
        }},
        '8': {'class_type': 'VAEDecode', 'inputs': {'samples': ['7', 0], 'vae': ['3', 0]}},
        '9': {'class_type': 'SaveImage', 'inputs': {'images': ['8', 0], 'filename_prefix': 'Qwen2_1_Output'}}
    }

    try:
        data = json.dumps({'prompt': workflow}).encode('utf-8')
        req = urllib.request.Request('http://127.0.0.1:8188/prompt', data=data, headers={'Content-Type': 'application/json'})
        with urllib.request.urlopen(req, timeout=15) as resp:
            resp_data = json.loads(resp.read().decode('utf-8'))
            prompt_id = resp_data['prompt_id']
    except urllib.error.HTTPError as e:
        err_msg = e.read().decode('utf-8')
        raise gr.Error(f'ComfyUI 요청 실패: {err_msg}')
    except Exception as e:
        raise gr.Error(f'요청 전송 실패: {e}')

    # 실시간 로그 기반 모니터링 루프
    filename, subfolder, type_ = None, None, None
    start_time = time.time()
    while time.time() - start_time < 600:
        time.sleep(1.5)
        elapsed = int(time.time() - start_time)
        
        # 백엔드 생존 체크
        if engine_proc.poll() is not None:
            tail = ''
            if os.path.exists(log_path):
                with open(log_path) as f:
                    tail = ''.join(f.readlines()[-25:])
            raise gr.Error(f'백엔드가 갑자기 중단되었습니다:\n{tail}')
        
        # 실시간 로그 상태 가져오기
        status_desc = get_latest_log_status()
        progress(None, desc=f'{status_desc} ({elapsed}초 경과)')
        
        try:
            with urllib.request.urlopen(f'http://127.0.0.1:8188/history/{prompt_id}', timeout=5) as resp:
                history = json.loads(resp.read().decode('utf-8'))
                if prompt_id in history:
                    outputs = history[prompt_id].get('outputs', {})
                    if '9' in outputs and 'images' in outputs['9']:
                        img_info = outputs['9']['images'][0]
                        filename = img_info['filename']
                        subfolder = img_info['subfolder']
                        type_ = img_info['type']
                        break
                    status = history[prompt_id].get('status', {})
                    if status.get('status_str') == 'error':
                        raise gr.Error(f'생성 실패: {status}')
        except gr.Error:
            raise
        except Exception:
            pass

    if not filename:
        tail = ''
        if os.path.exists(log_path):
            with open(log_path) as f:
                tail = ''.join(f.readlines()[-20:])
        raise gr.Error(f'이미지 생성 시간 초과 또는 오류가 발생했습니다. 로그:\n{tail}')

    progress(1.0, desc='✨ 이미지 생성 완료! 화면에 표시하는 중...')
    params = urllib.parse.urlencode({'filename': filename, 'subfolder': subfolder, 'type': type_})
    with urllib.request.urlopen(f'http://127.0.0.1:8188/view?{params}', timeout=15) as resp:
        return Image.open(io.BytesIO(resp.read()))

# 4. Gradio 웹 UI 생성
with gr.Blocks(theme=gr.themes.Soft(), title='Qwen-Image-2.1 Uncensored 검증기') as demo:
    gr.Markdown('# 🚀 Qwen-Image-2.1 Uncensored (Q4_0 GGUF) 실시간 고속 검증기')
    gr.Markdown('💡 **안내**: 16GB SWAP 메모리 설정 및 Dynamic VRAM 가속이 적용되었습니다. 첫 실행 시 가중치 로드(약 1분) 후 실시간으로 스텝이 올라갑니다.')
    
    with gr.Row():
        with gr.Column(scale=1):
            prompt_box = gr.Textbox(
                label='📝 프롬프트 (그릴 내용)',
                placeholder='영어로 프롬프트 입력',
                lines=4,
                value='a beautiful anime girl with long silver hair in a cherry blossom garden, sunny day, highly detailed, 8k masterpiece'
            )
            neg_prompt_box = gr.Textbox(
                label='🚫 부정 프롬프트 (제외할 내용)',
                lines=2,
                value='low quality, blurry, distorted, deformed, bad anatomy, worst quality'
            )
            
            with gr.Accordion('⚙️ 상세 옵션 (해상도 및 스텝 조절)', open=True):
                with gr.Row():
                    width_slider = gr.Slider(512, 1024, value=768, step=64, label='가로 해상도 (빠른 검증: 768 / 정밀: 1024)')
                    height_slider = gr.Slider(512, 1024, value=768, step=64, label='세로 해상도 (빠른 검증: 768 / 정밀: 1024)')
                steps_slider = gr.Slider(10, 30, value=20, step=1, label='생성 스텝수 (권장 20)')
                cfg_slider = gr.Slider(1.0, 7.0, value=1.0, step=0.5, label='CFG (Qwen-Image 공식 권장 1.0)')
                seed_input = gr.Number(value=-1, label='시드 (-1은 랜덤)')
            
            generate_btn = gr.Button('🚀 Qwen-Image 2.1 이미지 생성하기', variant='primary', size='lg')
            
        with gr.Column(scale=1):
            output_img = gr.Image(label='🖼️ 생성된 Qwen-Image 결과물', type='pil', interactive=False)
            
    generate_btn.click(
        fn=generate_image,
        inputs=[prompt_box, neg_prompt_box, steps_slider, cfg_slider, width_slider, height_slider, seed_input],
        outputs=output_img
    )

demo.queue().launch(share=True, debug=False)
